# 01.1 Tensor Basics

This notebook is the formal starting point for learning PyTorch. A tensor is the basic object that will hold features, labels, model parameters, losses, and intermediate activations. If tensor shape, dtype, and device are unclear, later topics such as model layers and training loops become much harder to debug.

Focus on reading tensors structurally. For every tensor, ask what each dimension means, what dtype it has, and whether it is on CPU or GPU.

## Learning Goals

After this notebook, you should be able to:

1. Create common tensors.
2. Read `shape`, `ndim`, `dtype`, and `device`.
3. Perform basic indexing and slicing.
4. Convert between `torch.Tensor` and `numpy.ndarray`.
5. Understand which operations share underlying storage.
6. Build the habit of checking shape before writing more code.

In [ ]:
import numpy as np
import torch

## What Is a Tensor?

In PyTorch, a tensor is the most fundamental data structure. You can initially think of it as a multi-dimensional array that also supports automatic differentiation and device movement. Like a NumPy array, it has shape, dtype, indexing, slicing, and broadcasting behavior. Unlike a plain NumPy array, it can live on a GPU and participate directly in gradient-based learning.

In [ ]:
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
cube = torch.arange(24).reshape(2, 3, 4)

print("scalar =", scalar)
print("vector =", vector)
print("matrix =\n", matrix)
print("cube.shape =", cube.shape)

## Creating Tensors

PyTorch gives you several ways to create tensors. `torch.tensor(...)` builds a tensor from explicit Python data. `torch.zeros`, `torch.ones`, and `torch.full` create tensors filled with known values. `torch.arange` creates a sequence of numbers, and `torch.randn` creates random values from a normal distribution.

Choose the constructor based on what you want the initial values to mean. For exercises and debugging, fixed values are usually easier to inspect than random values.

In [ ]:
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
steps = torch.arange(0, 10, 2)
noise = torch.randn(2, 4)
filled = torch.full((2, 2), fill_value=7.0)

print("zeros =\n", zeros)
print("ones =\n", ones)
print("steps =", steps)
print("noise =\n", noise)
print("filled =\n", filled)

In [ ]:
# Exercise 1
#
# Create a tensor named x.
#
# Requirements:
# - x should have shape (3, 4).
# - Every value should be 5.
# - The dtype should be torch.float32.
#
# After creating x, print the tensor, then print x.shape and x.dtype.

# x =
# print(x)
# print(x.shape, x.dtype)

In [ ]:
# Exercise 1 Reference Solution

x = torch.full((3, 4), 5.0, dtype=torch.float32)
print(x)
print(x.shape, x.dtype)

## Shape, Dimension, Dtype, Device

These are the four tensor properties you will inspect constantly. `shape` tells you the size of each dimension. `ndim` tells you how many dimensions exist. `dtype` tells you whether values are floats, integers, booleans, and so on. `device` tells you where the tensor lives, such as CPU or CUDA GPU.

Most PyTorch errors become easier to read if you first print these four properties for your inputs, targets, and model outputs.

In [ ]:
x = torch.randn(2, 3, dtype=torch.float32)

print("x =\n", x)
print("x.shape / shape =", x.shape)
print("x.ndim / ndim =", x.ndim)
print("x.dtype / dtype =", x.dtype)
print("x.device / device =", x.device)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
y = torch.arange(6, dtype=torch.float32).reshape(2, 3).to(device)

print("available device / available device:", device)
print("y.device =", y.device)

It is completely normal if you do not have a GPU available right now. The important concept is that tensors and models have a location. If the model is on the GPU but the input tensor is still on the CPU, PyTorch cannot run the operation. Later training code will consistently move both model and batch tensors to the same device.

## Indexing and Slicing

Basic indexing in `PyTorch` is very similar to `NumPy`.


In [ ]:
x = torch.arange(1, 13).reshape(3, 4)

print("x =\n", x)
print("first row / first row:", x[0])
print("last column / last column:", x[:, -1])
print("center 2x2 / center 2x2:\n", x[0:2, 1:3])

In [ ]:
# Exercise 2
#
# Practice indexing with the 4x4 tensor below.
#
# Fill in:
# - rows: rows 1 and 3 from grid
# - block: the top-left 2x2 block
# - evens: all even numbers from grid, selected with a boolean mask
#
# Print each result so you can verify that each indexing style did something
# different.

grid = torch.arange(16).reshape(4, 4)

# rows =
# block =
# evens =

# print(grid)
# print(rows)
# print(block)
# print(evens)

In [ ]:
# Exercise 2 Reference Solution

grid = torch.arange(16).reshape(4, 4)
rows = grid[[1, 3]]
block = grid[:2, :2]
evens = grid[grid % 2 == 0]

print(grid)
print(rows)
print(block)
print(evens)

## Converting to and from NumPy

Many projects use Pandas or NumPy for preprocessing and PyTorch for modeling, so conversion matters. `torch.from_numpy(arr)` creates a tensor view over a NumPy array when possible. That can be efficient, but it can also mean the tensor and array share memory.

When memory is shared, changing the NumPy array can change the tensor. If you need an independent tensor, clone it.

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
tensor_from_np = torch.from_numpy(arr)

print("NumPy -> Tensor:\n", tensor_from_np)

arr[0, 0] = 99.0
print("after modifying NumPy / after modifying NumPy:\n", tensor_from_np)

tensor_cpu = torch.tensor([[10.0, 20.0], [30.0, 40.0]], dtype=torch.float32)
arr_from_tensor = tensor_cpu.numpy()
tensor_cpu[0, 1] = -5.0

print("Tensor -> NumPy:\n", arr_from_tensor)

The example above shows that some `NumPy <-> Tensor` conversions share underlying memory.

If you do not want sharing, common options are:

- `tensor.clone()`
- `np.copy(...)`

In [ ]:
# Exercise 3
#
# Create one tensor that shares memory with a NumPy array and one tensor that
# does not.
#
# Steps:
# - Use torch.from_numpy(arr) to create t1.
# - Use clone() to create t2 from t1.
# - Change arr[0] to 100.0.
# - Print t1 and t2.
#
# Expected idea:
# - t1 should reflect the NumPy change because it shares memory.
# - t2 should keep the old values because it is a clone.

arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)

# t1 =
# t2 =
# arr[0] = 100.0
# print(t1)
# print(t2)

In [ ]:
# Exercise 3 Reference Solution

arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)
t1 = torch.from_numpy(arr)
t2 = t1.clone()
arr[0] = 100.0

print("shared version / shared version:", t1)
print("cloned version / cloned version:", t2)

## Common Type Conversions

Wrong dtypes often cause direct runtime errors. Feature tensors usually use `float32` because neural network layers perform floating-point math. Classification labels usually use `torch.long` because losses such as `CrossEntropyLoss` expect integer class indices. Boolean masks should use `bool` because they represent true/false selection conditions.

When an operation fails even though the shape looks correct, check dtype next.

In [ ]:
features = torch.tensor([[1, 2], [3, 4]], dtype=torch.int32)
labels = torch.tensor([0, 1, 1], dtype=torch.int32)

features_f = features.float()
labels_l = labels.long()

print("features_f.dtype =", features_f.dtype)
print("labels_l.dtype =", labels_l.dtype)

In [ ]:
# Exercise 4
#
# Practice dtype conversion.
#
# Fill in:
# - a_float: convert a to float32 values.
# - b_long: convert b to integer class-index style values with dtype torch.long.
#
# Print each tensor and its dtype. Notice that converting floats to long drops
# the decimal part.

a = torch.tensor([1, 2, 3])
b = torch.tensor([1.2, 2.8, 3.5])

# a_float =
# b_long =
# print(a_float, a_float.dtype)
# print(b_long, b_long.dtype)

In [ ]:
# Exercise 4 Reference Solution

a = torch.tensor([1, 2, 3])
b = torch.tensor([1.2, 2.8, 3.5])

a_float = a.float()
b_long = b.long()

print(a_float, a_float.dtype)
print(b_long, b_long.dtype)

## Summary

The key outcome is not the number of APIs you have seen, but a few foundational judgments:

- how many dimensions does this tensor have?
- what is its shape?
- is the dtype correct?
- what device is it on?
- does it share memory with NumPy?

You should now be able to answer:

1. What are the key similarities and differences between a `tensor` and an `ndarray`?
2. Why are `shape`, `dtype`, and `device` inspected so often?
3. Why do some `NumPy <-> Tensor` conversions affect each other?

Suggested next step:

- Move to `01_02_tensor_shape_ops.ipynb` to learn how tensors are reshaped and aligned.